In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# OPTION A (ON-THE-FLY) — OEM-SAFE CANONICAL (Finish_01..)
# FIXES:
# - Auto-detect ALL finish-patterns near target T (uses Version if exists)
# - Pick BEST pattern (most rows covered) OR you can choose another idx
# - Rewrite Finish -> Finish_01..Finish_05/06 IN-MEMORY (no new CSV)
# - Build db ONLY from the chosen pattern (prevents Finish4/Finish5 mixing)
# - Per-pass Offset interpolation (v4) + strict decreasing enforcement
# - Continuous: linear interp; Discrete (PRG/ADC/etc): nearest thickness (no interp)
# - HARD RULES: Material=1, Water(pass1=6 else 8), Mode/Content by pass
# - PASS LIMIT: if T>=50 => 5 passes (Finish_01..Finish_05), else 6
# - Export Makino-style XLSX exactly in 5 or 6 columns depending on T
# ============================================================

!pip -q install openpyxl

import numpy as np
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill, Border, Side
from openpyxl.utils import get_column_letter


In [3]:
# =========================
# 0) USER SETTINGS
# =========================
PATH = "/content/drive/MyDrive/Colab Notebooks/Makino/DS_U3,U6_BS0.20_St_Both Away Precision_DS.csv"

T_MAX = 150
T_EXTRAP_MAX = 160

In [4]:
# =========================
# 1) LOAD + CLEAN
# =========================
df = pd.read_csv(PATH)

def normalize_finish(f):
    f = str(f).strip()
    # keep as-is except trim (we will auto-map patterns later)
    return f

df["Finish"] = df["Finish"].apply(normalize_finish)

# Thickness clean
df["Thickness"] = (
    df["Thickness"].astype(str)
      .str.replace("mm", "", regex=False)
      .str.replace(" ", "", regex=False)
)
df["Thickness"] = pd.to_numeric(df["Thickness"], errors="coerce")
df = df.dropna(subset=["Thickness"])
df["Thickness"] = df["Thickness"].astype(int)

# Ensure Pass_NumPass numeric
df["Pass_NumPass"] = pd.to_numeric(df["Pass_NumPass"], errors="coerce")
df = df.dropna(subset=["Pass_NumPass"])
df["Pass_NumPass"] = df["Pass_NumPass"].astype(int)

In [5]:
# =========================
# 2) THICKNESS POLICY
# =========================
def thickness_status(T):
    if T <= T_MAX:
        return "SAFE_INTERPOLATED"
    if T <= T_EXTRAP_MAX:
        return "CAUTION_EXTRAPOLATED"
    raise ValueError("Thickness > 160 blocked in this version.")

def interp(y1, y2, T, T1, T2):
    if T1 == T2:
        return float(y1)
    return float(y1 + (y2 - y1) * (T - T1) / (T2 - T1))

def nearest(T, arr):
    arr = np.asarray(arr)
    return float(arr[np.argmin(np.abs(arr - T))])

def max_stage_allowed(T):
    return 6 if T < 50 else 5

In [6]:
# =========================
# 3) HARD RULES
# =========================
PASS_COUNT = {
    "Finish_01": 1,
    "Finish_02": 2,
    "Finish_03": 3,
    "Finish_04": 4,
    "Finish_05": 5,
    "Finish_06": 6,
}

def mode_for_pass(k):
    if k == 1: return 0
    if k in (2, 3): return 1
    return 3

def content_for_pass(k):
    return 1 if k == 1 else 0

def material_for_pass(k):
    return 1

def water_for_pass(k):
    return 6 if k == 1 else 8

In [7]:
# =========================
# 4) AUTO PATTERN DISCOVERY (Option A)
# =========================
def stage_from_maxpass(maxp: int) -> str:
    return f"Finish_{maxp:02d}"

def build_all_alias_dicts_for_T(df_raw: pd.DataFrame, T: int):
    """
    Build ALL possible alias dicts at nearest thickness T_ref.
    Uses Version if exists; otherwise one group only.
    Returns: (alias_list, T_ref)
    """
    Ts = np.array(sorted(df_raw["Thickness"].unique()))
    if len(Ts) == 0:
        raise ValueError("No thickness values in df_raw.")
    T_ref = int(Ts[np.argmin(np.abs(Ts - T))])

    sub = df_raw[df_raw["Thickness"] == T_ref].copy()
    if sub.empty:
        raise ValueError(f"No data for thickness={T_ref}")

    group_cols = ["Version"] if "Version" in sub.columns else None
    if group_cols is None:
        groups = [(None, sub)]
    else:
        groups = list(sub.groupby(group_cols, dropna=False))

    alias_dicts = []
    for gid, g in groups:
        fin_sum = (
            g.groupby("Finish")
             .agg(maxp=("Pass_NumPass", "max"),
                  freq=("Finish", "size"))
             .reset_index()
        )

        fin_sum["stage"] = fin_sum["maxp"].apply(stage_from_maxpass)

        # choose dominant finish per stage by frequency
        chosen = (
            fin_sum.sort_values(["stage", "freq"], ascending=[True, False])
                   .groupby("stage", as_index=False)
                   .first()
        )
        aliases = {row["stage"]: row["Finish"] for _, row in chosen.iterrows()}

        # apply stage limit rule by T (5 or 6)
        max_stage = max_stage_allowed(T)
        aliases = {k: v for k, v in aliases.items() if int(k.split("_")[1]) <= max_stage}

        # enforce continuous stages from 01 upward (stop at first missing)
        final = {}
        for i in range(1, max_stage + 1):
            key = f"Finish_{i:02d}"
            if key in aliases:
                final[key] = aliases[key]
            else:
                break

        if len(final) >= 2:
            alias_dicts.append(final)

    # unique dicts
    uniq = []
    seen = set()
    for d in alias_dicts:
        key = tuple(sorted(d.items()))
        if key not in seen:
            seen.add(key)
            uniq.append(d)

    if len(uniq) == 0:
        raise ValueError("Could not infer any finish-pattern dictionaries. Check Version/Finish labels.")

    return uniq, T_ref

def score_alias(df_at_Tref: pd.DataFrame, aliases: dict) -> int:
    real_names = set(aliases.values())
    return int(df_at_Tref[df_at_Tref["Finish"].isin(real_names)].shape[0])

def choose_best_alias(df_raw: pd.DataFrame, T_ref: int, alias_list: list):
    sub = df_raw[df_raw["Thickness"] == T_ref].copy()
    scored = [(i, score_alias(sub, a)) for i, a in enumerate(alias_list)]
    scored.sort(key=lambda x: x[1], reverse=True)
    best_idx = scored[0][0]
    return best_idx, scored

def apply_aliases_to_df(df_raw: pd.DataFrame, aliases: dict) -> pd.DataFrame:
    """
    Keep only rows in chosen pattern and create canonical Finish stages.
    """
    inv = {real: canon for canon, real in aliases.items()}
    out = df_raw.copy()
    out["Finish_real"] = out["Finish"]
    out["Finish_canon"] = out["Finish_real"].map(inv)
    out = out.dropna(subset=["Finish_canon"]).copy()
    out["Finish_canon"] = out["Finish_canon"].astype(str)
    out["Finish"] = out["Finish_canon"]  # overwrite Finish
    return out

# ---- choose pattern for this T
thickness_status(T)
alias_list, T_ref = build_all_alias_dicts_for_T(df, T)
best_idx, scored = choose_best_alias(df, T_ref, alias_list)

print("Nearest thickness template T_ref =", T_ref)
print("Found patterns =", len(alias_list))
print("Pattern scores (idx, covered_rows) =", scored)
print("Using BEST pattern idx =", best_idx)
print("Chosen aliases =", alias_list[best_idx])

aliases = alias_list[best_idx]
df_use = apply_aliases_to_df(df, aliases)

NameError: name 'T' is not defined

In [ ]:
# =========================
# 5) COLUMN TYPES (continuous vs discrete)
# =========================
KEY_COLS = ["Thickness", "Finish", "Pass_NumPass"]
CODE_PREFIXES = ("PRG", "ADC")
CODE_COLS = {"SVMode", "SVNo", "SM-Ref"}  # Makino code-ish string fields

continuous_cols, discrete_cols = [], []
for c in df_use.columns:
    if c in KEY_COLS or c in ("Finish_real", "Finish_canon"):
        continue
    if (c in CODE_COLS) or str(c).startswith(CODE_PREFIXES) or (df_use[c].dtype == "object"):
        discrete_cols.append(c)
    else:
        continuous_cols.append(c)

# Offset handled separately
if "Offset" in continuous_cols:
    continuous_cols.remove("Offset")

# Coerce continuous numeric
for c in continuous_cols + ["Offset"]:
    if c in df_use.columns:
        df_use[c] = pd.to_numeric(df_use[c], errors="coerce")

In [ ]:
# =========================
# 6) AGGREGATE -> DB (median continuous, mode discrete)
# =========================
def mode_series(s):
    s = s.dropna()
    if len(s) == 0:
        return np.nan
    return s.value_counts().idxmax()

agg = {c: "median" for c in continuous_cols}
agg.update({c: mode_series for c in discrete_cols if c in df_use.columns})

# always include these if present
if "Offset" in df_use.columns:
    agg["Offset"] = "median"
if "Mode" in df_use.columns:
    agg["Mode"] = mode_series
if "Content" in df_use.columns:
    agg["Content"] = mode_series

db = df_use.groupby(["Thickness", "Finish", "Pass_NumPass"], as_index=False).agg(agg)

In [ ]:
# =========================
# 7) v4 OFFSET — PER PASS INTERP + STRICTLY DECREASING
# =========================
def predict_offset_per_pass(db, T, finish, pass_k):
    thickness_status(T)
    sub = db[(db["Finish"] == finish) & (db["Pass_NumPass"] == pass_k)][["Thickness", "Offset"]].dropna()
    if sub.empty:
        raise ValueError(f"No Offset data for finish={finish}, pass={pass_k}")

    sub = sub.sort_values("Thickness")
    Ts = sub["Thickness"].to_numpy()
    Os = sub["Offset"].to_numpy()

    if T in Ts:
        # exact
        idx = np.where(Ts == T)[0][0]
        return float(Os[idx])

    if T < Ts[0]:
        return float(Os[0])

    if T <= Ts[-1]:
        i = np.searchsorted(Ts, T)
        T1, T2 = Ts[i-1], Ts[i]
        O1, O2 = Os[i-1], Os[i]
        return interp(O1, O2, T, T1, T2)

    # bounded extrap (last two points)
    if len(Ts) < 2:
        return float(Os[-1])
    T1, T2 = Ts[-2], Ts[-1]
    O1, O2 = Os[-2], Os[-1]
    return interp(O1, O2, T, T1, T2)

def generate_offsets_v4(db, T, finish, passes, eps=0.0005):
    offsets = {k: predict_offset_per_pass(db, T, finish, k) for k in range(1, passes+1)}
    for k in range(2, passes+1):
        if offsets[k] >= offsets[k-1]:
            offsets[k] = offsets[k-1] - eps
    return offsets

In [ ]:
# =========================
# 8) GENERATE ONE FINISH RECIPE (canonical Finish_01..)
# =========================
def generate_finish(T, finish):
    thickness_status(T)
    if finish not in PASS_COUNT:
        raise ValueError(f"Unknown finish: {finish}")

    passes = min(PASS_COUNT[finish], max_stage_allowed(T))
    rows = []

    for k in range(1, passes + 1):
        sub = db[(db["Finish"] == finish) & (db["Pass_NumPass"] == k)].copy()
        if sub.empty:
            raise ValueError(f"No data for finish={finish}, pass={k}")

        sub["Thickness"] = pd.to_numeric(sub["Thickness"], errors="coerce")
        sub = sub.dropna(subset=["Thickness"]).sort_values("Thickness")
        Ts = sub["Thickness"].to_numpy()
        if len(Ts) == 0:
            raise ValueError(f"No valid thickness values for finish={finish}, pass={k}")

        # exact-first, else interpolate, else bounded extrap/clip
        if T in Ts:
            T1 = T2 = T
        else:
            if T < Ts[0]:
                T1 = T2 = Ts[0]
            elif T > Ts[-1]:
                T1, T2 = (Ts[-2], Ts[-1]) if len(Ts) >= 2 else (Ts[-1], Ts[-1])
            else:
                i = np.searchsorted(Ts, T)
                T1, T2 = Ts[i - 1], Ts[i]

        r1 = sub[sub["Thickness"] == T1].iloc[0]
        r2 = sub[sub["Thickness"] == T2].iloc[0]

        out = {
            "Thickness": T,
            "Finish": finish,
            "Pass_NumPass": k,
            "Status": thickness_status(T),
            "Mode": mode_for_pass(k),
            "Content": content_for_pass(k),
            "Material": material_for_pass(k),
            "WATER": water_for_pass(k),
        }

        # continuous
        for c in continuous_cols:
            if c in sub.columns:
                out[c] = interp(r1[c], r2[c], T, T1, T2)

        # discrete (nearest thickness within <=150; exact-first)
        T_lookup = min(T, T_MAX)
        if T_lookup in Ts:
            Tn = T_lookup
        else:
            Tn = nearest(T_lookup, Ts)

        rn = sub[sub["Thickness"] == Tn].iloc[0]
        for c in discrete_cols:
            if c in sub.columns:
                out[c] = rn[c]

        rows.append(out)

    recipe = pd.DataFrame(rows)

    # v4 offsets
    offs = generate_offsets_v4(db, T, finish, passes)
    recipe["Offset"] = recipe["Pass_NumPass"].map(offs)

    return recipe

def generate_all_finishes(T):
    finishes = [f"Finish_{i:02d}" for i in range(1, max_stage_allowed(T) + 1)]
    all_recipes = {}
    for f in finishes:
        # only if exists in db
        if f in db["Finish"].unique():
            all_recipes[f] = generate_finish(T, f)
    return all_recipes

In [ ]:
# =========================
# 9) MAKINO XLSX EXPORT (5 or 6 columns)
# =========================
thin = Side(style="thin", color="000000")
border_all = Border(left=thin, right=thin, top=thin, bottom=thin)
fill_yellow = PatternFill("solid", fgColor="FFF59D")
fill_header = PatternFill("solid", fgColor="E0E0E0")
center = Alignment(horizontal="center", vertical="center", wrap_text=True)
left   = Alignment(horizontal="left", vertical="center", wrap_text=True)

def set_cell(ws, r, c, val, fill=None, font=None, align=None):
    cell = ws.cell(row=r, column=c, value=val)
    cell.border = border_all
    if fill: cell.fill = fill
    if font: cell.font = font
    if align: cell.alignment = align
    return cell

def merge_and_set(ws, r1, c1, r2, c2, val, fill=None, font=None, align=None):
    ws.merge_cells(start_row=r1, start_column=c1, end_row=r2, end_column=c2)
    set_cell(ws, r1, c1, val, fill=fill, font=font, align=align)

# Makino parameter order
PARAM_ORDER = [
    ("01:E No.", "E No."),
    ("02:WIRE DIA.", "WireDia"),
    ("03:MATERIAL", "Material"),
    ("04:THICKNESS", "Thickness"),
    ("05:CONTENT", "Content"),
    ("06:MODE", "Mode"),
    ("07:ONA", "ONA"),
    ("08:ONB", "ONB"),
    ("09:ONC", "ONC"),
    ("10:OND", "OND"),
    ("11:OFF", "OFF"),
    ("12:TS", "TS"),
    ("13:SCT", "SCT"),
    ("14:RCT", "RCT"),
    ("15:DCHG-S", "DCHG-S"),
    ("16:DCHG-R", "DCHG-R"),
    ("17:SV", "SV"),
    ("18:RV", "RV"),
    ("19:IPM", "IPM"),
    ("20:IPS", "IPS"),
    ("21:SV. MODE", "SVMode"),
    ("22:SV. No.", "SVNo"),
    ("23:SV. ADJ", "SVAdj"),
    ("24:SPEED", "Speed"),
    ("25:SM-REF", "SM-Ref"),
    ("26:PRG-ON", "PRG-ON"),
    ("27:PRG0-1", "PRG0-1"),
    ("28:PRG0-2", "PRG0-2"),
    ("29:PRG0-3", "PRG0-3"),
    ("30:PRG0-4", "PRG0-4"),
    ("31:PRG1-1", "PRG1-1"),
    ("32:PRG2-1", "PRG2-1"),
    ("33:PRG2-2", "PRG2-2"),
    ("34:PRG2-3", "PRG2-3"),
    ("35:PRG2-4", "PRG2-4"),
    ("36:PRG3-1", "PRG3-1"),
    ("37:PRG3-2", "PRG3-2"),
    ("38:PRG3-3", "PRG3-3"),
    ("39:PRG3-4", "PRG3-4"),
    ("40:PRG4-1", "PRG4-1"),
    ("41:PRG4-2", "PRG4-2"),
    ("42:PRG5-1", "PRG5-1"),
    ("43:PRG5-2", "PRG5-2"),
    ("44:PRG6-1", "PRG6-1"),
    ("45:PRG6-2", "PRG6-2"),
    ("46:ADC-ON", "ADC-ON"),
    ("47:ADC0-1", "ADC0-1"),
    ("48:ADC0-2", "ADC0-2"),
    ("49:ADC1-1", "ADC1-1"),
    ("50:ADC1-2", "ADC1-2"),
    ("51:ADC2-1", "ADC2-1"),
    ("52:ADC2-2", "ADC2-2"),
    ("53:ADC3-1", "ADC3-1"),
    ("54:ADC3-2", "ADC3-2"),
    ("55:ADC4-1", "ADC4-1"),
    ("56:ADC4-2", "ADC4-2"),
    ("57:ADC5-1", "ADC5-1"),
    ("58:ADC6-1", "ADC6-1"),
    ("59:ADC6-2", "ADC6-2"),
    ("60:WATER", "WATER"),
    ("61:OVERRIDE UPPER", "OverrideU"),
    ("62:OVERRIDE LOWER", "OverrideL"),
    ("63:WIRE SPEED", "WireSpeed"),
    ("64:W. TENSION", "WTension"),
    ("65:CONDUCTIVITY", "Conductivity"),
    ("66:EST. SPEED", "EstSpeed"),
]

def to_hexH(v, width=None):
    if pd.isna(v) or v == "":
        return ""
    if isinstance(v, str) and v.strip().upper().endswith("H"):
        return v.strip().upper()
    try:
        iv = int(round(float(v)))
    except:
        return v
    if width is None:
        width = 2 if iv <= 0xFF else 4
    return f"{iv:0{width}X}H"

# NOTE: ADC0-2 should often be shown as plain "1" (not "01H") in your Makino screenshots,
# so we DO NOT force hex for ADC0-2.
HEX_4 = {"PRG0-1"}
HEX_2 = {
    "PRG-ON","SM-Ref","SVMode","SVNo","ADC-ON","ADC0-1",
    "PRG0-2","PRG0-3","PRG0-4",
    "PRG1-1","PRG2-3","PRG2-4","PRG3-1","PRG3-2","PRG3-4",
    "PRG4-1","PRG4-2","PRG5-1","PRG5-2","PRG6-1","PRG6-2",
    "ADC1-1","ADC1-2","ADC2-1","ADC2-2","ADC3-1","ADC3-2","ADC4-1","ADC4-2","ADC5-1","ADC6-1","ADC6-2"
}

def format_cell_value(colname, v):
    if colname == "E No.":
        if pd.isna(v) or v == "":
            return ""
        if isinstance(v, str) and v.strip().upper().startswith("E"):
            return v.strip().upper()
        try:
            return f"E{int(round(float(v)))}"
        except:
            return v

    if colname in HEX_4:
        return to_hexH(v, width=4)
    if colname in HEX_2:
        return to_hexH(v, width=2)

    # numeric formatting
    if isinstance(v, (float, int, np.floating, np.integer)):
        vv = float(v)
        return round(vv, 4) if abs(vv) < 1000 else vv

    return v

def build_makino_final(T, all_recipes, out_path):
    """
    - Columns count = 5 if T>=50 else 6
    - Top block shows OFFSETS per stage up to its pass count
    - Bottom 01..66 uses FINAL stage recipe (Finish_05 or Finish_06) pass1..N
    """
    ncols = max_stage_allowed(T)
    final_stage = f"Finish_{ncols:02d}"
    if final_stage not in all_recipes:
        raise ValueError(f"Missing final stage recipe: {final_stage}")

    base = all_recipes[final_stage].set_index("Pass_NumPass")  # authoritative final settings

    wb = Workbook()
    ws = wb.active
    ws.title = f"T{T}"

    # widths
    ws.column_dimensions["A"].width = 24
    ws.column_dimensions["B"].width = 18
    for j in range(1, ncols + 1):
        ws.column_dimensions[get_column_letter(2 + j)].width = 14

    PASS_LABELS = {1:"1st",2:"2nd",3:"3rd",4:"4th",5:"5th",6:"6th"}

    r = 1
    merge_and_set(ws, r, 1, r, 2, "W. Thickness (mm)", fill=fill_header, font=Font(bold=True), align=left)
    merge_and_set(ws, r, 3, r, 2 + ncols, T, align=center); r += 1

    merge_and_set(ws, r, 1, r, 2, "Angle", fill=fill_header, font=Font(bold=True), align=left)
    merge_and_set(ws, r, 3, r, 2 + ncols, 0, align=center); r += 1

    merge_and_set(ws, r, 1, r, 2, "Process", fill=fill_header, font=Font(bold=True), align=left)
    for j in range(1, ncols + 1):
        set_cell(ws, r, 2 + j, PASS_LABELS[j], fill=fill_header, font=Font(bold=True), align=center)
    r += 1

    # ---- TOP OFFSET BLOCK (Finish_01..Finish_05/06) ----
    for i in range(1, ncols + 1):
        stage = f"Finish_{i:02d}"
        merge_and_set(ws, r, 1, r, 2, stage, fill=fill_header, font=Font(bold=True), align=left)

        if stage in all_recipes:
            rec = all_recipes[stage].set_index("Pass_NumPass")
            maxp = int(rec.index.max())
            for j in range(1, ncols + 1):
                if j <= maxp:
                    val = rec.loc[j, "Offset"]
                    set_cell(ws, r, 2 + j, round(float(val), 4), fill=fill_yellow, align=center)
                else:
                    set_cell(ws, r, 2 + j, "", align=center)
        else:
            for j in range(1, ncols + 1):
                set_cell(ws, r, 2 + j, "", align=center)

        r += 1

    # spacer
    merge_and_set(ws, r, 1, r, 2 + ncols, "", align=center); r += 1

    # ---- BOTTOM 01..66 ----
    for label, colname in PARAM_ORDER:
        merge_and_set(ws, r, 1, r, 2, label, fill=fill_header, font=Font(bold=True), align=left)
        for j in range(1, ncols + 1):
            if (j not in base.index) or (colname not in base.columns):
                set_cell(ws, r, 2 + j, "", align=center)
                continue
            v = base.loc[j, colname]
            v = format_cell_value(colname, v)
            set_cell(ws, r, 2 + j, v, fill=fill_yellow, align=center)
        r += 1

    ws.freeze_panes = "C4"
    wb.save(out_path)
    print("Saved:", out_path)

In [ ]:
# =========================
# 10) RUN
# =========================
T =5  # <-- change your target thickness here (<=160 allowed)

all_recipes = generate_all_finishes(T)
out_path = f"/content/Makino_Final_T{T}_OptionA.xlsx"
build_makino_final(T, all_recipes, out_path)

print("DONE:", out_path)


Saved: /content/Makino_Final_T5_OptionA.xlsx
DONE: /content/Makino_Final_T5_OptionA.xlsx
